<a href="https://colab.research.google.com/github/koushiksr/CI-CD_Dev_Test_Docker_Diploy/blob/main/tool_calling_llm.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install transformers torch

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 94.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 97.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 56.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 12.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 8.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 6.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 70.4 MB/s eta 0:00:00
  Attempting uninstall: nvidia-nvjitlink-cu12
    Found existing installation: nvidia-nvjitlink-cu12 12.5.82
    Uninstalling nvidia-nvjitlink-cu12-12.5.82:
      Successfully uninstalled nvidia-nvjitlin

In [ ]:
import json
import uuid
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

# Define helper functions for weather
def get_current_temperature(location: str, unit: str) -> float:
    """
    Simulates getting the current temperature at a location.
    Args:
        location: The location in "City, Country" format.
        unit: The unit ("celsius" or "fahrenheit").
    Returns:
        Dummy temperature (22.0) for demo.
    """
    print(f"Tool call: get_current_temperature(location='{location}', unit='{unit}')")
    return 22.0  # Replace with actual API call (e.g., OpenWeatherMap)

def get_current_wind_speed(location: str) -> float:
    """
    Simulates getting the current wind speed at a location.
    Args:
        location: The location in "City, Country" format.
    Returns:
        Dummy wind speed (6.0) for demo.
    """
    print(f"Tool call: get_current_wind_speed(location='{location}')")
    return 6.0  # Replace with actual API call

# Define tool list with function schemas
tools = [
    {
        "name": "get_current_temperature",
        "description": "Get the current temperature at a location",
        "parameters": {
            "type": "object",
            "properties": {
                "location": {"type": "string", "description": "City, Country"},
                "unit": {"type": "string", "description": "celsius or fahrenheit"}
            },
            "required": ["location", "unit"]
        }
    },
    {
        "name": "get_current_wind_speed",
        "description": "Get the current wind speed at a location",
        "parameters": {
            "type": "object",
            "properties": {
                "location": {"type": "string", "description": "City, Country"}
            },
            "required": ["location"]
        }
    }
]

# Load model and tokenizer (replace with a model that supports tool calling)
model_name = "microsoft/Phi-3-medium-4k-instruct"  # Use a tool-calling-capable model
# It seems there was a misunderstanding about the token usage. For publicly available models like Phi-3,
# a token is generally not required for `from_pretrained`. If you were trying to access a private
# model or needed to authenticate for other reasons, you would typically log in using `huggingface_cli login`
# or pass the token to the `from_pretrained` function like this:
# tokenizer = AutoTokenizer.from_pretrained(model_name, token="your_hf_token")
# model = AutoModelForCausalLM.from_pretrained(model_name, token="your_hf_token", device_map="auto")

# For this publicly available model, we don't need the token parameter for basic loading.
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name, device_map="auto")
model.eval()

# Set up conversation history
messages = [
    {"role": "system", "content": "You are a bot that responds to weather queries. Reply with the unit used in the queried location."},
    {"role": "user", "content": "Hey, what's the temperature in Paris right now?"}
]

# Function to execute tool calls
def execute_tool(tool_call):
    tool_name = tool_call["function"]["name"]
    args = tool_call["function"]["arguments"]
    # Ensure arguments is a dictionary
    if not isinstance(args, dict):
        try:
            args = json.loads(args)
        except json.JSONDecodeError:
            print(f"Could not parse arguments for tool {tool_name}: {args}")
            return "Error: Could not parse tool arguments"

    if tool_name == "get_current_temperature":
        return get_current_temperature(args.get("location"), args.get("unit"))
    elif tool_name == "get_current_wind_speed":
        return get_current_wind_speed(args.get("location"))
    return "Unknown tool"

# Main function to handle tool calling
def handle_tool_calling(messages, tools):
    # Prepare model input with chat template
    # The Phi-3 model uses the 'chatml' template for tool calling
    inputs = tokenizer.apply_chat_template(
        messages,
        chat_template="chatml",  # Use chatml template for Phi-3
        tools=tools,
        add_generation_prompt=True,
        return_dict=True,
        return_tensors="pt"
    )

    # Move inputs to GPU if available
    inputs = {k: v.to(model.device) for k, v in inputs.items()}

    # Generate LLM response
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=128)

    # Decode output
    # The Phi-3 model outputs tool calls in a specific format within the chatml template
    decoded_output = tokenizer.decode(out[0][len(inputs["input_ids"][0]):], skip_special_tokens=False) # Don't skip special tokens to parse tool calls

    # Parse potential tool call from the chatml output
    # Tool calls in Phi-3 ChatML template are within <tool_code> and </tool_code> tags
    tool_call_start = decoded_output.find("<tool_code>")
    tool_call_end = decoded_output.find("</tool_code>")

    if tool_call_start != -1 and tool_call_end != -1:
        tool_call_json_str = decoded_output[tool_call_start + len("<tool_code>") : tool_call_end].strip()
        try:
            tool_call = json.loads(tool_call_json_str)
            if "tool_code" in tool_call:
                 # Assuming the tool call structure is like {"tool_code": "function_name(arg=value)"} or similar
                 # We need to parse this string to extract the function name and arguments
                 # A more robust approach would involve a dedicated tool parsing library or regex
                 # For this example, let's assume a simple format and try to extract
                 # This is a simplification and might need adjustment based on actual model output
                 import re
                 match = re.match(r"(\w+)\((.*)\)", tool_call.get("tool_code", ""))
                 if match:
                     function_name = match.group(1)
                     arguments_str = match.group(2)
                     # Parse arguments string into a dictionary (simple case)
                     args_dict = {}
                     for arg in arguments_str.split(','):
                         if '=' in arg:
                             key, value = arg.split('=')
                             args_dict[key.strip()] = value.strip().strip('"').strip("'") # Basic stripping

                     # Construct a tool_call dictionary that fits the execute_tool function structure
                     tool_call_for_execution = {
                         "function": {
                             "name": function_name,
                             "arguments": args_dict
                         }
                     }

                     tool_call_id = str(uuid.uuid4())

                     # Add tool call to messages
                     messages.append({
                         "role": "assistant",
                         "tool_calls": [{"id": tool_call_id, "type": "function", "function": tool_call_for_execution["function"]}]
                     })

                     # Execute tool
                     result = execute_tool(tool_call_for_execution)

                     # Add tool result to messages
                     messages.append({
                         "role": "tool",
                         "tool_call_id": tool_call_id,
                         "name": tool_call_for_execution["function"]["name"],
                         "content": str(result)
                     })

                     # Generate final response with updated history
                     inputs = tokenizer.apply_chat_template(
                         messages,
                         chat_template="chatml", # Use chatml template
                         tools=tools,
                         add_generation_prompt=True,
                         return_dict=True,
                         return_tensors="pt"
                     )
                     inputs = {k: v.to(model.device) for k, v in inputs.items()}

                     with torch.no_grad():
                         final_out = model.generate(**inputs, max_new_tokens=128)

                     return tokenizer.decode(final_out[0][len(inputs["input_ids"][0]):], skip_special_tokens=True)
                 else:
                     print(f"Could not parse tool code format: {tool_call.get('tool_code')}")
                     return decoded_output # Return original output if parsing fails
            else:
                 print("Tool call JSON does not contain 'tool_code'")
                 return decoded_output # Return original output if parsing fails

        except json.JSONDecodeError:
            print("Could not decode tool call JSON.")
            return decoded_output # Return original output if JSON is invalid

    # If no tool call detected, return the raw decoded output
    return decoded_output

# Run the tool-calling pipeline
result = handle_tool_calling(messages, tools)
print(result)

Fetching 6 files:   0%|          | 0/6 [00:00<?, ?it/s]

model-00001-of-00006.safetensors:   0%|          | 0.00/4.92G [00:00<?, ?B/s]

model-00005-of-00006.safetensors:   0%|          | 0.00/4.77G [00:00<?, ?B/s]

model-00004-of-00006.safetensors:   0%|          | 0.00/4.77G [00:00<?, ?B/s]

model-00003-of-00006.safetensors:   0%|          | 0.00/4.90G [00:00<?, ?B/s]

model-00006-of-00006.safetensors:   0%|          | 0.00/3.61G [00:00<?, ?B/s]

model-00002-of-00006.safetensors:   0%|          | 0.00/4.95G [00:00<?, ?B/s]

KeyboardInterrupt: 